In [1]:
import pandas as pd

USEFUL_COLUMNS = ['Name', 'Sex', 'Age', 'BodyweightKg', 'Equipment',
                   'Best3SquatKg', 'Best3BenchKg', 'Best3DeadliftKg',
                   'TotalKg', 'Date']

power_data = pd.read_csv("data/openpowerlifting.csv", usecols = USEFUL_COLUMNS)


In [2]:
power_data = power_data[power_data['Equipment'] == 'Raw'].copy()


In [3]:
power_data = power_data.dropna(subset=['Age', 'BodyweightKg']).copy()


In [4]:
power_data = power_data.sort_values(['Name', 'Date'])

power_data = power_data[power_data['Sex'] != 'Mx'].copy()
power_data['Sex_encoded'] = power_data['Sex'].map({'M': 0, 'F': 1})

In [5]:
squat_data = power_data[power_data['Best3SquatKg'] > 0].copy()
squat_data = squat_data.sort_values(['Name', 'Date'])
squat_data['experience'] = squat_data.groupby('Name').cumcount() + 1

squat_data[['Name', 'Date', 'Best3SquatKg', 'Sex_encoded', 'experience']].head(10)

,Name,Date,Best3SquatKg,Sex_encoded,experience
1400040,A Aanvi,2026-05-15,77.5,1,1
289339,A Ajeesha,2017-12-04,112.5,1,1
282960,A Ashwin,2012-12-10,170.0,0,1
2482449,A Belousov,2019-10-01,75.0,0,1
1413739,A Hemanth Kumar,2022-08-10,175.0,0,1
152513,A I Temaki,2023-12-17,375.0,0,1
484698,A I Temaki,2024-04-13,370.0,0,2
485081,A I Temaki,2025-03-29,400.0,0,3
2809445,A Jay Montanez,2016-02-27,225.0,0,1
1398505,A K S Shri Ram,2019-09-26,117.5,0,1


In [6]:
bench_data = power_data[power_data['Best3BenchKg'] > 0].copy()
bench_data = bench_data.sort_values(['Name', 'Date'])
bench_data['experience'] = bench_data.groupby('Name').cumcount() + 1

bench_data[['Name', 'Date', 'Best3BenchKg', 'Sex_encoded', 'experience']].head(10)

,Name,Date,Best3BenchKg,Sex_encoded,experience
1400040,A Aanvi,2026-05-15,40.0,1,1
289339,A Ajeesha,2017-12-04,55.0,1,1
1402214,A Arun Kumar,2018-11-14,135.0,0,1
282960,A Ashwin,2012-12-10,95.0,0,1
2482449,A Belousov,2019-10-01,75.0,0,1
3732954,A Cavatorta,2004-06-19,110.0,0,1
1413739,A Hemanth Kumar,2022-08-10,105.0,0,1
484698,A I Temaki,2024-04-13,212.0,0,1
485081,A I Temaki,2025-03-29,220.0,0,2
331027,A J Ceronio,2016-12-04,55.0,0,1


In [7]:
deadlift_data = power_data[power_data['Best3DeadliftKg'] > 0].copy()
deadlift_data = deadlift_data.sort_values(['Name', 'Date'])
deadlift_data['experience'] = deadlift_data.groupby('Name').cumcount() + 1

deadlift_data[['Name', 'Date', 'Best3DeadliftKg', 'Sex_encoded', 'experience']].head(10)

,Name,Date,Best3DeadliftKg,Sex_encoded,experience
1400040,A Aanvi,2026-05-15,90.0,1,1
289339,A Ajeesha,2017-12-04,132.5,1,1
282960,A Ashwin,2012-12-10,220.0,0,1
2482449,A Belousov,2019-10-01,100.0,0,1
1413739,A Hemanth Kumar,2022-08-10,200.0,0,1
484698,A I Temaki,2024-04-13,235.0,0,1
485081,A I Temaki,2025-03-29,235.0,0,2
331027,A J Ceronio,2016-12-04,130.0,0,1
2809445,A Jay Montanez,2016-02-27,270.0,0,1
1398505,A K S Shri Ram,2019-09-26,150.0,0,1


In [8]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def train_lift_model(data, target_column):
    FEATURES = ['Age', 'BodyweightKg', 'Sex_encoded', 'experience']
    X = data[FEATURES]
    y = data[target_column]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = Ridge()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    naive_prediction = y_train.mean()
    naive_mae = mean_absolute_error(y_test, [naive_prediction] * len(y_test))

    print(f"{target_column} — Naive MAE: {naive_mae:.2f}, Model MAE: {mae:.2f}, Model RMSE: {rmse:.2f}")

    return model

In [9]:
squat_model = train_lift_model(squat_data, 'Best3SquatKg')
bench_model = train_lift_model(bench_data, 'Best3BenchKg')
deadlift_model = train_lift_model(deadlift_data, 'Best3DeadliftKg')

Best3SquatKg — Naive MAE: 48.65, Model MAE: 27.24, Model RMSE: 35.86
Best3BenchKg — Naive MAE: 38.25, Model MAE: 20.50, Model RMSE: 26.93
Best3DeadliftKg — Naive MAE: 50.47, Model MAE: 28.92, Model RMSE: 38.07


In [10]:
import joblib

joblib.dump(squat_model, 'squat_population_model.joblib')
joblib.dump(bench_model, 'bench_population_model.joblib')
joblib.dump(deadlift_model, 'deadlift_population_model.joblib')

print("Saved")

Saved
